# .NET RAG Blueprint Quickstart

This notebook validates the .NET RAG services with unversioned API routes. It is derived from the Python Workbench quickstart workflow, but targets the .NET ingestor, RAG server, and optional reranker service directly.

## 1. Configuration

Set these values to match your local runtime. ChromaDB is the default vector database for local .NET development; switch `VDB_ENDPOINT` to Milvus when validating Milvus parity.

In [ ]:
import json
import os
import tempfile
from pathlib import Path

import requests

RAG_BASE_URL = os.getenv("DOTNET_RAG_BASE_URL", "http://localhost:8081")
INGESTOR_BASE_URL = os.getenv("DOTNET_INGESTOR_BASE_URL", "http://localhost:8082")
RERANKER_BASE_URL = os.getenv("DOTNET_RERANKER_BASE_URL", "http://localhost:8083")

# ChromaDB default. For local Milvus parity, use: http://localhost:19530
VDB_ENDPOINT = os.getenv("DOTNET_VDB_ENDPOINT", "http://localhost:8000")
COLLECTION_NAME = os.getenv("DOTNET_TEST_COLLECTION", "dotnet_notebook_quickstart")

LLM_MODEL = os.getenv("DOTNET_LLM_MODEL", "nemotron-3-nano:4b")
EMBEDDING_MODEL = os.getenv("DOTNET_EMBEDDING_MODEL", "nomic-embed-text")
RERANKER_MODEL = os.getenv("DOTNET_RERANKER_MODEL", EMBEDDING_MODEL)

print(json.dumps({
    "rag": RAG_BASE_URL,
    "ingestor": INGESTOR_BASE_URL,
    "reranker": RERANKER_BASE_URL,
    "vdb_endpoint": VDB_ENDPOINT,
    "collection": COLLECTION_NAME,
}, indent=2))

In [ ]:
def show_response(response):
    print(response.request.method, response.url)
    print("status:", response.status_code)
    content_type = response.headers.get("content-type", "")
    if "application/json" in content_type:
        print(json.dumps(response.json(), indent=2))
    else:
        print(response.text[:2000])
    response.raise_for_status()
    return response


def chat_content(response_json):
    return (
        response_json.get("choices", [{}])[0]
        .get("message", {})
        .get("content", "")
    )


def normalize_search_results(response_json):
    results = response_json.get("results", [])
    normalized = []
    for item in results:
        content = item.get("content", "")
        if isinstance(content, list):
            content = "\n".join(part.get("text", "") for part in content)
        attrs = item.get("attributes", {}) or {}
        normalized.append({
            "document_name": item.get("document_name") or item.get("filename") or attrs.get("filename"),
            "score": item.get("score"),
            "content": content,
            "attributes": attrs,
        })
    return normalized

## 2. Health Checks

In [ ]:
show_response(requests.get(f"{RAG_BASE_URL}/health", timeout=30))
show_response(requests.get(f"{INGESTOR_BASE_URL}/health", timeout=30))
show_response(requests.get(f"{RAG_BASE_URL}/configuration", timeout=30))

## 3. Collection Lifecycle

In [ ]:
collection_params = {
    "vdb_endpoint": VDB_ENDPOINT,
    "collection_type": "text",
    "embedding_dimension": 768,
}

show_response(requests.post(
    f"{INGESTOR_BASE_URL}/collections",
    params=collection_params,
    json=[COLLECTION_NAME],
    timeout=60,
))

show_response(requests.get(
    f"{INGESTOR_BASE_URL}/collections",
    params={"vdb_endpoint": VDB_ENDPOINT},
    timeout=60,
))

## 4. Upload and List Documents

In [ ]:
sample_text = """NVIDIA RAG Blueprint .NET notebook parity fixture.
The 2022 FIFA World Cup was won by Argentina.
This sentence is intentionally deterministic for retrieval validation.
"""

tmpdir = Path(tempfile.mkdtemp(prefix="dotnet-rag-notebook-"))
sample_path = tmpdir / "dotnet_notebook_fixture.txt"
sample_path.write_text(sample_text, encoding="utf-8")

upload_payload = {
    "vdb_endpoint": VDB_ENDPOINT,
    "collection_name": COLLECTION_NAME,
    "blocking": True,
    "generate_summary": False,
    "split_options": {"chunk_size": 1024, "chunk_overlap": 150},
    "custom_metadata": [
        {"filename": sample_path.name, "metadata": {"category": "notebook-test"}}
    ],
    "documents_catalog_metadata": [
        {"filename": sample_path.name, "description": "Notebook parity fixture", "tags": ["dotnet", "parity"]}
    ],
}

with sample_path.open("rb") as handle:
    response = requests.post(
        f"{INGESTOR_BASE_URL}/documents",
        files={"documents": (sample_path.name, handle, "text/plain")},
        data={"data": json.dumps(upload_payload)},
        timeout=180,
    )
show_response(response)

show_response(requests.get(
    f"{INGESTOR_BASE_URL}/documents",
    params={"collection_name": COLLECTION_NAME, "vdb_endpoint": VDB_ENDPOINT, "force_get_metadata": "true"},
    timeout=60,
))

## 5. Chat Without Knowledge Base

In [ ]:
payload = {
    "messages": [{"role": "user", "content": "Say hello in one sentence."}],
    "use_knowledge_base": False,
    "temperature": 0.2,
    "top_p": 0.7,
    "max_tokens": 128,
    "model": LLM_MODEL,
}

response = show_response(requests.post(f"{RAG_BASE_URL}/chat/completions", json=payload, timeout=180))
print("content:", chat_content(response.json()))

## 6. Chat With Knowledge Base

In [ ]:
payload = {
    "messages": [{"role": "user", "content": "Who won the 2022 FIFA World Cup?"}],
    "use_knowledge_base": True,
    "temperature": 0.2,
    "top_p": 0.7,
    "max_tokens": 256,
    "reranker_top_k": 5,
    "vdb_top_k": 20,
    "vdb_endpoint": VDB_ENDPOINT,
    "collection_names": [COLLECTION_NAME],
    "enable_query_rewriting": False,
    "enable_reranker": True,
    "enable_guardrails": False,
    "enable_citations": True,
    "model": LLM_MODEL,
    "embedding_model": EMBEDDING_MODEL,
    "reranker_model": RERANKER_MODEL,
}

response = show_response(requests.post(f"{RAG_BASE_URL}/chat/completions", json=payload, timeout=180))
print("content:", chat_content(response.json()))

## 7. Search With and Without Reranking

In [ ]:
base_search_payload = {
    "query": "Who won the 2022 FIFA World Cup?",
    "messages": [{"role": "user", "content": "Who won the 2022 FIFA World Cup?"}],
    "reranker_top_k": 3,
    "vdb_top_k": 20,
    "vdb_endpoint": VDB_ENDPOINT,
    "collection_names": [COLLECTION_NAME],
    "enable_query_rewriting": False,
    "embedding_model": EMBEDDING_MODEL,
    "reranker_model": RERANKER_MODEL,
}

for enabled in [False, True]:
    print("\nreranker enabled:", enabled)
    payload = {**base_search_payload, "enable_reranker": enabled}
    response = show_response(requests.post(f"{RAG_BASE_URL}/search", json=payload, timeout=180))
    for result in normalize_search_results(response.json()):
        print(json.dumps(result, indent=2)[:1500])

## 8. Optional Summary Check

This returns pending when the document was uploaded without summary generation. Re-run the upload cell with `generate_summary` set to `True` to validate summary retrieval.

In [ ]:
response = requests.get(
    f"{RAG_BASE_URL}/summary",
    params={"collection_name": COLLECTION_NAME, "file_name": sample_path.name, "blocking": "false"},
    timeout=60,
)
show_response(response)

## 9. Cleanup

In [ ]:
show_response(requests.delete(
    f"{INGESTOR_BASE_URL}/documents",
    params={"collection_name": COLLECTION_NAME, "vdb_endpoint": VDB_ENDPOINT},
    json=[sample_path.name],
    timeout=60,
))

show_response(requests.delete(
    f"{INGESTOR_BASE_URL}/collections",
    params={"vdb_endpoint": VDB_ENDPOINT},
    json=[COLLECTION_NAME],
    timeout=60,
))